In [5]:
# ============================ §1 配置 ============================
import time

import numpy as np
import pandas as pd

PKL = r"D:\2026_Summer\TradingApp\data\v3.0\all_symbol_min_full_main_close_k_1.pkl"
OUTDIR = r"D:\2026_Summer\TradingApp\data\v3.0"

# ---- 筛查阈值（自己调）----
TH_JUMP = 0.05        # C10: 相邻「真实相邻分钟」收盘价跳变超过这个比例就算异常
TH_MINUTE_ACTIVE = 0.20   # C8: 某分钟出现在 >= 20% 的交易日里，才算这个品种的正常交易分钟
TH_STALE_DAYS = 180      # C2: 最后一根 bar 距全表最新日期超过这么多天 = 停更
TH_MIN_TRADING_DAYS = 250    # ★: 交易日数少于这个 = 历史太短
TH_MIN_MEDIAN_VOLUME = 10000  # ★: 日成交量中位数低于这个 = 流动性不足
FP_SINCE = "2023-01-01"   # C8: 「当前 session 指纹」从这天算起（避开历史制度换挡）
TH_K_BAD_PCT = 0.00     # ★: K 坏值占比超过这个 = 不可复权
TH_FAKE_DAY_PCT = 0.20    # ★: 假日历日占比超过这个 = 大半历史是补出来的

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
print("配置就绪")

配置就绪


In [6]:
# ============================ §2 加载（约 10s / 常驻 ~6.4GB）============================
t0 = time.time()
df = pd.read_pickle(PKL)
print(f"loaded in {time.time() - t0:.1f}s   shape={df.shape}")
print(f"常驻内存(shallow) = {df.memory_usage(deep=False).sum() / 1e9:.2f} GB")

loaded in 7.9s   shape=(66284866, 11)
常驻内存(shallow) = 6.36 GB


## C0 全局结构

这一格只是把「表长什么样」摊开，不做判断。

In [7]:
print("=" * 70, "\nC0 全局结构\n", "=" * 70)
print(f"shape (rows, cols) : {df.shape}")
print(f"index name / dtype : {df.index.name} / {df.index.dtype}")
print(f"index tz           : {getattr(df.index, 'tz', None)}   <- None = tz-naive(北京墙钟)")
print(f"index min / max    : {df.index.min()}  ..  {df.index.max()}")
print(f"index.is_unique    : {df.index.is_unique}        <- 全表 False 是正常的(长表堆叠)")
print(f"index.is_monotonic : {df.index.is_monotonic_increasing}   <- 全表 False 是正常的(按品种分块)")
print(f"不同时间戳个数     : {df.index.nunique():,}")
print(f"不同 trading_date  : {df['trading_date'].nunique():,}")
print(f"不同 underlying_sym: {df['underlying_symbol'].nunique():,}")
print(f"不同 contract      : {df['contract'].nunique():,}")
print("\ndtypes:")
print(df.dtypes.to_string())
print("\nhead(3):")
print(df.head(3).to_string())
print("\ntail(3):")
print(df.tail(3).to_string())

C0 全局结构
shape (rows, cols) : (66284866, 11)
index name / dtype : datetime / datetime64[ns]
index tz           : None   <- None = tz-naive(北京墙钟)
index min / max    : 2010-01-04 09:00:00  ..  2026-07-29 15:15:00
index.is_unique    : False        <- 全表 False 是正常的(长表堆叠)
index.is_monotonic : False   <- 全表 False 是正常的(按品种分块)
不同时间戳个数     : 2,155,760
不同 trading_date  : 4,017
不同 underlying_sym: 79
不同 contract      : 4,041

dtypes:
trading_date         datetime64[ns]
close                       float64
open                        float64
high                        float64
low                         float64
volume                      float64
total_turnover              float64
open_interest               float64
contract                     object
underlying_symbol            object
K                           float64

head(3):
                    trading_date   close    open    high     low  volume  total_turnover  open_interest contract underlying_symbol         K
datetime                    

## §2 基础量（后面每一格都依赖这一格）

全部用 numpy 一维数组 + `np.bincount` 做逐品种聚合，避免在 6600 万行上做
pandas 多级 groupby。三个工具函数：

- `cnt(mask)` → 每个品种里 mask 为真的行数
- `g(values, how)` → 每个品种的 min/max/sum/...
- `tbl(**cols)` → 把若干个「长度 NS 的数组」拼成一张以品种为行的表

In [4]:
sym_codes, SYMS = pd.factorize(df["underlying_symbol"])
sym_codes = sym_codes.astype(np.int32)
NS = len(SYMS)

con_codes, CONS = pd.factorize(df["contract"])
con_codes = con_codes.astype(np.int32)

IDX_NS = df.index.values.astype("int64")                       # 每根 bar 时间戳(ns)
TOD = ((IDX_NS // 60_000_000_000) % 1440).astype(np.int16)     # minute-of-day 0..1439
CAL_D = (IDX_NS // 86_400_000_000_000).astype(np.int32)        # 日历日(epoch 天序号)
TD_D = df["trading_date"].values.astype("datetime64[D]").astype(np.int32)  # trading_date 天序号

CD0, TD0 = int(CAL_D.min()), int(TD_D.min())
NCD, NTD = int(CAL_D.max()) - CD0 + 1, int(TD_D.max()) - TD0 + 1

# (品种, trading_date) 复合键 —— C6/C7/C8b/C11 都靠它
KEY_TD = sym_codes.astype(np.int64) * NTD + (TD_D - TD0)
# (品种, 日历日) 复合键 —— C6 判「整个日历日是假的」
KEY_CD = sym_codes.astype(np.int64) * NCD + (CAL_D - CD0)

N_BARS = np.bincount(sym_codes, minlength=NS).astype(np.int64)


def cnt(mask) -> np.ndarray:
    """每个品种里 mask 为真的行数（下标 = 品种 code）。"""
    return np.bincount(sym_codes[mask], minlength=NS).astype(np.int64)


def g(values, how) -> np.ndarray:
    """每个品种的聚合值。how: 'min'/'max'/'sum'/'mean'/'nunique'..."""
    return pd.Series(values).groupby(sym_codes).agg(how).sort_index().to_numpy()


def tbl(**cols) -> pd.DataFrame:
    """长度 NS 的数组 -> 以品种为行的表。"""
    return pd.DataFrame(cols, index=pd.Index(SYMS, name="sym"))


# 品种是否按块连续存放 + 块内时间是否严格递增。
# C9/C10 用「相邻行相减」判换月和跳变，必须先确认这个前提成立。
SAME_PREV = np.empty(len(df), dtype=bool)
SAME_PREV[0] = False
SAME_PREV[1:] = sym_codes[1:] == sym_codes[:-1]
DT_NS = np.empty(len(df), dtype="int64")
DT_NS[0] = 0
DT_NS[1:] = np.diff(IDX_NS)

BLOCKY = bool(np.all(np.diff(sym_codes) >= 0))
IN_BLOCK_SORTED = bool(np.all(DT_NS[SAME_PREV] > 0))
print(f"NS (品种数)            = {NS}")
print(f"品种按块连续存放       = {BLOCKY}")
print(f"块内时间严格递增       = {IN_BLOCK_SORTED}   <- 必须 True，C9/C10 才成立")
print(f"总行数校验             = {N_BARS.sum():,} vs {len(df):,}")

NS (品种数)            = 79
品种按块连续存放       = True
块内时间严格递增       = False   <- 必须 True，C9/C10 才成立
总行数校验             = 66,284,866 vs 66,284,866


## C1 单品种索引完整性

全表 index 不唯一是设计使然（同一时刻每个品种各一行）。**真正要查的是：同一个
品种内部有没有重复时间戳、有没有时间倒流。**

`dup_adjacent` 用相邻行判重（快）；`dup_exact` 用逐品种 nunique 判重（慢但严格，
能抓到不相邻的重复）。两者应该一致。

In [8]:
dup_adjacent = cnt(SAME_PREV & (DT_NS == 0))
backwards = cnt(SAME_PREV & (DT_NS < 0))

t0 = time.time()
n_uniq_ts = pd.Series(IDX_NS).groupby(sym_codes).nunique().sort_index().to_numpy()
print(f"(逐品种 nunique 耗时 {time.time() - t0:.1f}s)")

C1 = tbl(
    n_bars=N_BARS,
    n_unique_ts=n_uniq_ts,
    dup_exact=N_BARS - n_uniq_ts,
    dup_adjacent=dup_adjacent,
    time_backwards=backwards,
)
print("=" * 70, "\nC1 单品种索引完整性\n", "=" * 70)
bad = C1[(C1["dup_exact"] > 0) | (C1["time_backwards"] > 0)]
print(f"有问题的品种数: {len(bad)} / {NS}")
print(bad.to_string() if len(bad) else "  (全部干净)")

(逐品种 nunique 耗时 8.7s)
C1 单品种索引完整性
有问题的品种数: 6 / 79
      n_bars  n_unique_ts  dup_exact  dup_adjacent  time_backwards
sym                                                               
JR    815634       667830     147804        147804               0
LR    760038       627150     132888        132888               0
PM    909198       880044      29154         29154               0
RI    908972       880044      28928         28928               0
WH   1021294       880044     141250        141250               0
ZC   1195605       989389     206216        206216               0


## C2 覆盖区间与数据 vintage

重点看 `last_ts` 的**聚集**：如果一批品种的最后一根 bar 都停在同一个较早的日期，
那不是"退市"，而是这份 pkl 由**多个不同时点的快照拼接**而成 —— 那批品种的历史
是残缺的，不能直接用。

In [9]:
first_ts = g(IDX_NS, "min")
last_ts = g(IDX_NS, "max")
n_td_per_sym = pd.Series(TD_D).groupby(sym_codes).nunique().sort_index().to_numpy()

DATA_MAX = pd.Timestamp(int(IDX_NS.max()))
C2 = tbl(
    n_bars=N_BARS,
    first_ts=pd.to_datetime(first_ts),
    last_ts=pd.to_datetime(last_ts),
    n_trading_days=n_td_per_sym,
)
C2["stale_days"] = (DATA_MAX - C2["last_ts"]).dt.days
C2["is_stale"] = C2["stale_days"] > TH_STALE_DAYS
C2["too_short"] = C2["n_trading_days"] < TH_MIN_TRADING_DAYS

print("=" * 70, "\nC2 覆盖区间\n", "=" * 70)
print(f"全表最新时间: {DATA_MAX}")
print("\n--- last_ts 按日期聚集（这就是 vintage 指纹）---")
vint = C2.groupby(C2["last_ts"].dt.date).agg(
    n_symbols=("n_bars", "size"), symbols=("n_bars", lambda s: " ".join(sorted(s.index)))
).sort_index()
print(vint.to_string())
print(f"\n停更品种 (stale_days > {TH_STALE_DAYS}): {int(C2['is_stale'].sum())}")
print(f"历史过短 (交易日 < {TH_MIN_TRADING_DAYS}): {int(C2['too_short'].sum())}")
print("\n--- 全表（按 bar 数降序）---")
print(C2.sort_values("n_bars", ascending=False).to_string())

C2 覆盖区间
全表最新时间: 2026-07-29 15:15:00

--- last_ts 按日期聚集（这就是 vintage 指纹）---
            n_symbols                                                                                                                                                                      symbols
last_ts                                                                                                                                                                                           
2024-04-22          9                                                                                                                                                   AO BR EC IM LC PX SH SI TL
2025-12-31          1                                                                                                                                                                           LR
2026-01-16          8                                                                                                                             

## C3 OHLC 合法性

五类违规，逐品种计数：

| 列 | 含义 |
|---|---|
| `nan_px` | open/high/low/close 里任一为 NaN |
| `nonpos_px` | 任一 <= 0（0 价就在这里） |
| `high_lt_low` | high < low |
| `high_not_max` | high < max(open, close) |
| `low_not_min` | low > min(open, close) |

In [23]:
o = df["open"].to_numpy()
h = df["high"].to_numpy()
lo = df["low"].to_numpy()
c = df["close"].to_numpy()

nan_px = np.isnan(o) | np.isnan(h) | np.isnan(lo) | np.isnan(c)
nonpos_px = (o <= 0) | (h <= 0) | (lo <= 0) | (c <= 0)
high_lt_low = h < lo
high_not_max = h < np.maximum(o, c)
low_not_min = lo > np.minimum(o, c)

C3 = tbl(
    n_bars=N_BARS,
    nan_px=cnt(nan_px),
    nonpos_px=cnt(nonpos_px),
    high_lt_low=cnt(high_lt_low),
    high_not_max=cnt(high_not_max),
    low_not_min=cnt(low_not_min),
    close_min=g(c, "min"),
    close_max=g(c, "max"),
)
C3["ohlc_bad_total"] = C3[["nan_px", "nonpos_px", "high_lt_low", "high_not_max", "low_not_min"]].sum(axis=1)

print("=" * 70, "\nC3 OHLC 合法性\n", "=" * 70)
print(f"全表: nan={int(nan_px.sum()):,}  nonpos={int(nonpos_px.sum()):,}  "
      f"h<l={int(high_lt_low.sum()):,}  h<max(o,c)={int(high_not_max.sum()):,}  "
      f"l>min(o,c)={int(low_not_min.sum()):,}")
bad3 = C3[C3["ohlc_bad_total"] > 0].sort_values("ohlc_bad_total", ascending=False)
print(f"\n有问题的品种数: {len(bad3)} / {NS}")
print(bad3.to_string() if len(bad3) else "  (全部干净)")

if nonpos_px.any():
    print("\n--- 0/负价样本（前 15 行）---")
    print(df.loc[nonpos_px, ["trading_date", "open", "high", "low", "close",
                             "volume", "contract", "underlying_symbol"]].head(15).to_string())

del nan_px, nonpos_px, high_lt_low, high_not_max, low_not_min

C3 OHLC 合法性
全表: nan=0  nonpos=96  h<l=0  h<max(o,c)=0  l>min(o,c)=0

有问题的品种数: 3 / 79
      n_bars  nan_px  nonpos_px  high_lt_low  high_not_max  low_not_min  close_min  close_max  ohlc_bad_total
sym                                                                                                          
B    1277250       0         91            0             0            0        0.0    5850.00              91
RR    570521       0          4            0             0            0     3279.0    3730.00               4
BB    664666       0          1            0             0            0        0.0     451.65               1

--- 0/负价样本（前 15 行）---
                    trading_date  open  high  low  close  volume contract underlying_symbol
datetime                                                                                   
2017-05-12 13:31:00   2017-05-12   0.0   0.0  0.0    0.0     1.0    B1709                 B
2017-05-12 13:32:00   2017-05-12   0.0   0.0  0.0    0.0     1.0  

## C4 成交量 / 成交额 / 持仓

`vol0_turn_pos` 和 `volpos_turn0` 是**自相矛盾**行：有量无额 / 有额无量。
这两类比单纯的 0 更值得警惕，因为它说明字段本身不可信。

In [11]:
v = df["volume"].to_numpy()
tn = df["total_turnover"].to_numpy()
oi = df["open_interest"].to_numpy()

C4 = tbl(
    n_bars=N_BARS,
    vol_nan=cnt(np.isnan(v)),
    vol_neg=cnt(v < 0),
    vol_zero=cnt(v == 0),
    turn_neg=cnt(tn < 0),
    turn_zero=cnt(tn == 0),
    oi_zero=cnt(oi == 0),
    oi_neg=cnt(oi < 0),
    vol0_turn_pos=cnt((v == 0) & (tn > 0)),
    volpos_turn0=cnt((v > 0) & (tn == 0)),
)
C4["vol_zero_pct"] = C4["vol_zero"] / C4["n_bars"]
C4["contradictory"] = C4["vol0_turn_pos"] + C4["volpos_turn0"]

print("=" * 70, "\nC4 成交量/成交额/持仓\n", "=" * 70)
print(f"全表: vol_neg={int((v < 0).sum()):,}  turn_neg={int((tn < 0).sum()):,}  "
      f"vol_zero={int((v == 0).sum()):,}  oi_zero={int((oi == 0).sum()):,}")
print("\n--- 有负值或自相矛盾的品种 ---")
bad4 = C4[(C4["vol_neg"] > 0) | (C4["turn_neg"] > 0) | (C4["contradictory"] > 0)]
print(bad4.to_string() if len(bad4) else "  (无)")
print("\n--- 零成交占比最高的 15 个品种 ---")
print(C4.sort_values("vol_zero_pct", ascending=False).head(15).to_string())

if (v < 0).any():
    print("\n--- 负成交量样本 ---")
    print(df.loc[v < 0, ["trading_date", "close", "volume", "total_turnover",
                         "contract", "underlying_symbol"]].to_string())

C4 成交量/成交额/持仓
全表: vol_neg=1  turn_neg=4,160  vol_zero=8,580,598  oi_zero=2,784,998

--- 有负值或自相矛盾的品种 ---
      n_bars  vol_nan  vol_neg  vol_zero  turn_neg  turn_zero  oi_zero  oi_neg  vol0_turn_pos  volpos_turn0  vol_zero_pct  contradictory
sym                                                                                                                                     
CF   1269000        0        0     14438         5      14438        0       0              0             0      0.011377              0
CY    753468        0        0    187757       273     187394       34       0            100             0      0.249190            100
JR    815634        0        0    803523       469     803054   479824       0              1             0      0.985151              1
LR    760038        0        0    730145       287     729852   497524       0              6             0      0.960669              6
MA   1168656        0        0     19781         4      19781        0    

## C5 K（复权因子）

K 是后面做复权的唯一依据。**0 会导致除零，inf 会污染整条价格序列，NaN 会静默产生
空洞。** 这一格的输出直接决定「哪些品种不能做复权」。

In [12]:
k = df["K"].to_numpy()
k_nan = np.isnan(k)
k_inf = np.isinf(k)
k_zero = k == 0
k_neg = k < 0
k_bad = k_nan | k_inf | (k <= 0)

k_fin = np.where(np.isfinite(k), k, np.nan)
C5 = tbl(
    n_bars=N_BARS,
    k_nan=cnt(k_nan),
    k_inf=cnt(k_inf),
    k_zero=cnt(k_zero),
    k_neg=cnt(k_neg),
    k_bad=cnt(k_bad),
    k_min=g(k_fin, "min"),
    k_max=g(k_fin, "max"),
)
C5["k_bad_pct"] = C5["k_bad"] / C5["n_bars"]

print("=" * 70, "\nC5 K 复权因子\n", "=" * 70)
print(f"全表: nan={int(k_nan.sum()):,}  inf={int(k_inf.sum()):,}  zero={int(k_zero.sum()):,}  "
      f"neg={int(k_neg.sum()):,}  合计坏行={int(k_bad.sum()):,} ({k_bad.mean():.2%})")
print("\n--- K 有坏值的品种（这些品种不能直接复权）---")
bad5 = C5[C5["k_bad"] > 0].sort_values("k_bad", ascending=False)
print(bad5.to_string() if len(bad5) else "  (无)")
print("\n--- 全表 K 分位数（仅有限值）---")
kk = k[np.isfinite(k)]
for q in [0, 0.001, 0.01, 0.25, 0.5, 0.75, 0.99, 0.999, 1]:
    print(f"  q{q:<6} {np.quantile(kk, q):.8g}")
del kk, k_nan, k_inf, k_zero, k_neg, k_fin

C5 K 复权因子
全表: nan=1,315,125  inf=44,748  zero=464,006  neg=0  合计坏行=1,823,879 (2.75%)

--- K 有坏值的品种（这些品种不能直接复权）---
      n_bars   k_nan  k_inf  k_zero  k_neg   k_bad     k_min     k_max  k_bad_pct
sym                                                                              
ZC   1195605  487965      0    1384      0  489349  0.000000  1.170740   0.409290
JR    815634  313914  44748       0      0  358662  0.959246  1.155493   0.439734
WH   1021294  343972      0     678      0  344650  0.000000  1.915581   0.337464
PM    909198       0      0  223514      0  223514  0.000000  2.466777   0.245836
RI    908972       0      0  223288      0  223288  0.000000  2.305994   0.245649
RS    714838  169274      0   15142      0  184416  0.000000  1.328992   0.257983

--- 全表 K 分位数（仅有限值）---
  q0      0
  q0.001  0
  q0.01   0.13507567
  q0.25   0.85584718
  q0.5    1.0254832
  q0.75   1.2179327
  q0.99   3.0406178
  q0.999  5.4919592
  q1      5.5140268


## C6 padding 假 bar

文件名里的 `min_full` = **全分钟网格**：休市的日盘/夜盘也会被补成一整段
`open==high==low==close` 且 `volume==0` 的 bar。

两个层次：
- `pad_bars` —— 单根 flat+零成交（可能只是冷门品种真的没成交，不一定是假的）
- `fake_cal_days` —— **整个日历日的成交量合计为 0** → 这一天肯定是补出来的

后者才是必须处理的：它会让 1D 聚合把两个日历日并成一根日线。

In [13]:
flat = (o == h) & (h == lo) & (lo == c)
pad = flat & (v == 0)

vol_by_cd = np.bincount(KEY_CD, weights=v, minlength=NS * NCD)
bars_by_cd = np.bincount(KEY_CD, minlength=NS * NCD)
fake_cd = (bars_by_cd > 0) & (vol_by_cd == 0)
fake_by_sym = np.bincount((np.flatnonzero(fake_cd) // NCD), minlength=NS).astype(np.int64)
fake_bars_by_sym = np.bincount(
    (np.flatnonzero(fake_cd) // NCD), weights=bars_by_cd[fake_cd], minlength=NS
).astype(np.int64)
n_cd_by_sym = np.bincount((np.flatnonzero(bars_by_cd > 0) // NCD), minlength=NS).astype(np.int64)

C6 = tbl(
    n_bars=N_BARS,
    flat_bars=cnt(flat),
    pad_bars=cnt(pad),
    n_calendar_days=n_cd_by_sym,
    fake_cal_days=fake_by_sym,
    fake_day_bars=fake_bars_by_sym,
)
C6["pad_pct"] = C6["pad_bars"] / C6["n_bars"]
C6["fake_day_pct"] = C6["fake_cal_days"] / C6["n_calendar_days"]

print("=" * 70, "\nC6 padding 假 bar\n", "=" * 70)
print(f"全表: flat={int(flat.sum()):,}  flat&vol==0={int(pad.sum()):,} ({pad.mean():.2%})")
print(f"整日成交量为 0 的 (品种,日历日) 组合: {int(fake_cd.sum()):,}")
print(f"其中包含的 bar 数: {int(bars_by_cd[fake_cd].sum()):,}")
print("\n--- 假日历日最多的 15 个品种 ---")
print(C6.sort_values("fake_cal_days", ascending=False).head(15).to_string())

print("\n--- 假日历日样例（RB，前 10 个）---")
rb = int(np.flatnonzero(SYMS == "RB")[0]) if (SYMS == "RB").any() else 0
rb_fake = np.flatnonzero(fake_cd[rb * NCD:(rb + 1) * NCD]) + CD0
print(f"品种 {SYMS[rb]}: 共 {len(rb_fake)} 个假日历日")
print(pd.to_datetime(rb_fake, unit="D")[:10].to_list())

C6 padding 假 bar
全表: flat=10,942,926  flat&vol==0=8,580,486 (12.94%)
整日成交量为 0 的 (品种,日历日) 组合: 16,958
其中包含的 bar 数: 4,561,132

--- 假日历日最多的 15 个品种 ---
      n_bars  flat_bars  pad_bars  n_calendar_days  fake_cal_days  fake_day_bars   pad_pct  fake_day_pct
sym                                                                                                     
PM    909198     893003    874029             3894           2746         649750  0.961319      0.705187
JR    815634     809798    803521             2955           2174         639128  0.985149      0.735702
LR    760038     741411    730142             2775           2148         618336  0.960665      0.774054
BB    664666     594185    581374             2941           1864         421264  0.874686      0.633798
RI    908972     742131    673903             3894           1851         447254  0.741390      0.475347
RS    714838     673493    652830             3163           1102         249052  0.913256      0.348403
WH   1021294 

## C7 trading_date 完整性

⚠️ **别用「一个 trading_date 覆盖几个日历日」这个绝对数当判据** —— 它取决于品种
自己的夜盘形态（实测众数只有 1 和 2 两种）：

- **无夜盘**（IF/IC/IH/T/TF/TS 等 31 个）：1 个日历日
- **有夜盘**（其余 48 个）：2 个日历日 = {D-1 晚, D}。注意**跨零点也还是 2 个**：
  AU 的 `21:00–23:59`(D-1) + `00:00–02:30`(D) + 日盘(D)，日历日仍只有 {D-1, D}

所以这里用**不依赖形态的判据**：*一个 trading_date 里是否混进了「整日零成交」的
日历日*（即 C6 的假日历日）。这才是 1D 聚合的直接杀手 —— 它会把休市日和真实交易日
并进同一根日线。

`max_cal_days` / `cal_days_over_mode` 保留为辅助诊断：超出该品种自己的众数，说明那天
的日历日构成异常（多半是 padding，也可能是历史上夜盘时段换过挡）。实测 max 可达 4，
全部来自长假 padding。

In [14]:
# 每个 (品种, trading_date) 覆盖了几个不同日历日
comb = (KEY_TD << 20) | (CAL_D - CD0).astype(np.int64)
u = np.unique(comb)
ncal_by_ktd = np.bincount((u >> 20), minlength=NS * NTD)
del comb, u

# --- 主判据：trading_date 里混进了假(整日零成交)日历日 ---
row_fake_cd = fake_cd[KEY_CD]                        # 行级：这一行落在假日历日里
td_has_fake = np.bincount(KEY_TD[row_fake_cd], minlength=NS * NTD) > 0
n_td_polluted = np.bincount(np.flatnonzero(td_has_fake) // NTD, minlength=NS).astype(np.int64)

# --- 辅助诊断：日历日数的众数 / 最大值 / 超众数的天数 ---
nz = np.flatnonzero(ncal_by_ktd > 0)
_nc = pd.DataFrame({"sym_code": nz // NTD, "ncal": ncal_by_ktd[nz]})
_agg = _nc.groupby("sym_code")["ncal"].agg(
    mode_ncal=lambda s: int(s.mode().iloc[0]), max_ncal="max"
).reindex(range(NS))
over_mode = _nc.merge(_agg["mode_ncal"].rename("m"), left_on="sym_code", right_index=True)
n_over_mode = (over_mode.assign(hit=lambda d: d["ncal"] > d["m"])
               .groupby("sym_code")["hit"].sum().reindex(range(NS), fill_value=0).to_numpy())

C7 = tbl(
    n_trading_days=n_td_per_sym,
    mode_cal_days=_agg["mode_ncal"].to_numpy(),
    max_cal_days=_agg["max_ncal"].to_numpy(),
    cal_days_over_mode=n_over_mode.astype(np.int64),
    td_polluted_by_fake_day=n_td_polluted,
)
C7["td_polluted_pct"] = C7["td_polluted_by_fake_day"] / C7["n_trading_days"]

print("=" * 70, "\nC7 trading_date 完整性\n", "=" * 70)
print(f"含假日历日的 (品种,trading_date) 组合: {int(td_has_fake.sum()):,}")
print("\n--- 主判据：被假日历日污染的 trading_date（降序）---")
bad7 = C7[C7["td_polluted_by_fake_day"] > 0].sort_values("td_polluted_pct", ascending=False)
print(bad7.to_string() if len(bad7) else "  (无)")
print("\n--- 辅助：日历日数众数分布（1=无夜盘 / 2=有夜盘，含跨零点。两者都正常）---")
for m, grp in C7.groupby("mode_cal_days"):
    print(f"  众数={m}  ({len(grp)} 个): {' '.join(sorted(grp.index))}")

print(f"\n--- 污染样例：{SYMS[rb]} ---")
rb_bad_td = np.flatnonzero(td_has_fake[rb * NTD:(rb + 1) * NTD]) + TD0
print(f"共 {len(rb_bad_td)} 个被污染的 trading_date")
if len(rb_bad_td):
    show_td = pd.Timestamp(pd.to_datetime(rb_bad_td[-1], unit="D"))
    day = df.loc[(df["underlying_symbol"] == SYMS[rb]) & (df["trading_date"] == show_td)]
    t = day.index.to_series()
    runs = (t.diff() > pd.Timedelta("1min")).cumsum()
    print(f"trading_date = {show_td.date()}   n_bars = {len(day)}")
    for rid, grp in t.groupby(runs):
        vsum = day.loc[grp.index, "volume"].sum()
        tag = "  <-- 假 session（整段零成交，是补出来的）" if vsum == 0 else ""
        print(f"  run {int(rid)}: {grp.iloc[0]} -> {grp.iloc[-1]}  "
              f"({len(grp)} bars, volume合计={vsum:,.0f}){tag}")

C7 trading_date 完整性
含假日历日的 (品种,trading_date) 组合: 16,895

--- 主判据：被假日历日污染的 trading_date（降序）---
     n_trading_days  mode_cal_days  max_cal_days  cal_days_over_mode  td_polluted_by_fake_day  td_polluted_pct
sym                                                                                                           
LR             2775              1             1                   0                     2148         0.774054
JR             2955              1             1                   0                     2174         0.735702
PM             3894              1             1                   0                     2746         0.705187
BB             2941              1             1                   0                     1864         0.633798
RI             3894              1             1                   0                     1851         0.475347
RS             3163              1             1                   0                     1102         0.348403
ZC             298

## C8 session 结构指纹

**不硬编码交易所日历**，直接从数据里数：某个 minute-of-day 在这个品种的多少个交易
日里出现过。出现率 >= `TH_MINUTE_ACTIVE` 的分钟，就是这个品种的真实交易分钟。

把这些分钟按连续段合并，就得到每个品种的 session 指纹 —— 这是后面做「时段内重采样」
的唯一依据。

In [15]:
def runs_of(mask_1440: np.ndarray) -> list[tuple[int, int]]:
    """把布尔分钟掩码合并成连续段 [(起分钟, 止分钟), ...]。"""
    idx = np.flatnonzero(mask_1440)
    if len(idx) == 0:
        return []
    brk = np.flatnonzero(np.diff(idx) > 1)
    starts = np.concatenate(([idx[0]], idx[brk + 1]))
    ends = np.concatenate((idx[brk], [idx[-1]]))
    return list(zip(starts.tolist(), ends.tolist()))


def fmt_m(m: int) -> str:
    return f"{m // 60:02d}:{m % 60:02d}"


def fingerprint(row_mask=None) -> pd.DataFrame:
    """在 row_mask 限定的行范围内，算每个品种的 session 指纹。row_mask=None 表示全历史。"""
    if row_mask is None:
        sc, td_, tod_ = sym_codes, TD_D, TOD
    else:
        sc, td_, tod_ = sym_codes[row_mask], TD_D[row_mask], TOD[row_mask]
    ndays = (pd.Series(td_).groupby(sc).nunique()
             .reindex(range(NS)).fillna(0).to_numpy())
    mh_ = np.bincount(sc.astype(np.int64) * 1440 + tod_, minlength=NS * 1440).reshape(NS, 1440)
    with np.errstate(invalid="ignore", divide="ignore"):
        act = mh_ / np.where(ndays > 0, ndays, np.nan)[:, None] >= TH_MINUTE_ACTIVE
    out = []
    for i, s in enumerate(SYMS):
        rr = runs_of(act[i])
        out.append({
            "sym": s,
            "n_days": int(ndays[i]),
            "n_active_minutes": int(act[i].sum()),
            "n_sessions": len(rr),
            # 夜盘 = 21:00 之后开始的段，或跨零点落在 00:00-03:00 的段
            "has_night": any(a >= 1140 or b < 240 for a, b in rr),
            "crosses_midnight": any(b < 240 for a, b in rr),
            "fingerprint": " | ".join(f"{fmt_m(a)}-{fmt_m(b)}({b - a + 1})" for a, b in rr),
        })
    return pd.DataFrame(out).set_index("sym")


# ⚠️ 全历史指纹是「历史上所有制度的并集」：出现率 >=20% 会把已经废弃的时段也算进来
# （例：RB 显示 09:00 起，但 09:00 那根竞价 bar 2023 年起就没了）。
# 所以同时算一份 FP_SINCE 之后的**当前指纹** —— 做重采样要用后者。
C8_all = fingerprint(None)
C8 = fingerprint(TD_D >= int(np.datetime64(FP_SINCE, "D").astype(int)))

print("=" * 70, "\nC8 session 指纹（出现率 >= %.0f%% 的分钟）\n" % (TH_MINUTE_ACTIVE * 100), "=" * 70)
print(f"\n>>> 当前指纹（{FP_SINCE} 起）—— 做重采样用这份")
print(C8[["n_days", "n_sessions", "has_night", "crosses_midnight", "fingerprint"]].to_string())
print("\n--- 按当前指纹分组：一共几种 session 结构 ---")
print(C8[C8["n_days"] > 0].groupby("fingerprint").agg(
    n_symbols=("n_sessions", "size"), symbols=("n_sessions", lambda s: " ".join(sorted(s.index)))
).sort_values("n_symbols", ascending=False).to_string())

print(f"\n>>> 指纹在 {FP_SINCE} 前后发生过变化的品种（全历史上线前必须逐个确认）")
chg_fp = C8_all.loc[(C8_all["fingerprint"] != C8["fingerprint"]) & (C8["n_days"] > 0),
                    ["fingerprint"]].rename(columns={"fingerprint": "全历史指纹"})
chg_fp["当前指纹"] = C8.loc[chg_fp.index, "fingerprint"]
print(chg_fp.to_string() if len(chg_fp) else "  (无变化)")

C8 session 指纹（出现率 >= 20% 的分钟）

>>> 当前指纹（2023-01-01 起）—— 做重采样用这份
     n_days  n_sessions  has_night  crosses_midnight                                                                                fingerprint
sym                                                                                                                                            
A       858           4       True             False                     09:01-10:15(75) | 10:31-11:30(60) | 13:31-15:00(90) | 21:00-23:00(121)
AG      858           5       True              True  00:00-02:30(151) | 09:01-10:15(75) | 10:31-11:30(60) | 13:31-15:00(90) | 21:00-23:59(180)
AL      858           5       True              True   00:00-01:00(61) | 09:01-10:15(75) | 10:31-11:30(60) | 13:31-15:00(90) | 21:00-23:59(180)
AO      200           5       True              True   00:00-01:00(61) | 09:01-10:15(75) | 10:31-11:30(60) | 13:31-15:00(90) | 21:00-23:59(180)
AP      858           3      False             False                    

### C8b 逐年 bar 数换挡

同一个品种在不同年份 bar 数会跳变（夜盘上线/调整、竞价 bar 位置变化）。
**全历史上线前必须先看这张表** —— 在近两年验过的规则，在老数据上可能静默失效。

In [16]:
bars_by_ktd = np.bincount(KEY_TD, minlength=NS * NTD)
nzk = np.flatnonzero(bars_by_ktd > 0)
per_td = pd.DataFrame({
    "sym": SYMS[nzk // NTD],
    "td": pd.to_datetime((nzk % NTD) + TD0, unit="D"),
    "n_bars": bars_by_ktd[nzk],
})
per_td["year"] = per_td["td"].dt.year

C8b = per_td.pivot_table(index="sym", columns="year", values="n_bars", aggfunc="median")
print("=" * 70, "\nC8b 逐年「每交易日 bar 数」中位数\n", "=" * 70)
print(C8b.to_string())

print("\n--- 每个品种出现过几种不同的 bars/day（>1 = 有换挡）---")
shift_tbl = per_td.groupby("sym")["n_bars"].agg(
    n_distinct="nunique", most_common=lambda s: int(s.mode().iloc[0]),
    min="min", max="max",
).sort_values("n_distinct", ascending=False)
print(shift_tbl.to_string())

C8b 逐年「每交易日 bar 数」中位数
year   2010   2011   2012   2013   2014   2015   2016   2017   2018   2019   2020   2021   2022   2023   2024   2025   2026
sym                                                                                                                        
A     226.0  226.0  226.0  226.0  226.0  376.0  376.0  376.0  376.0  346.0  346.0  346.0  346.0  346.0  346.0  346.0  346.0
AG      NaN    NaN  226.0  391.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0
AL    226.0  226.0  226.0  226.0  466.0  466.0  466.0  466.0  466.0  466.0  466.0  466.0  466.0  466.0  466.0  466.0  466.0
AO      NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN  466.0  466.0    NaN    NaN
AP      NaN    NaN    NaN    NaN    NaN    NaN    NaN  226.0  226.0  226.0  226.0  226.0  226.0  226.0  226.0  226.0  226.0
AU    226.0  226.0  226.0  391.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0  556.0 

## C9 合约与换月

未复权价在换月处**必然跳空，这是预期行为，不是 bug**。这一格量化它，
并检查换月是否都发生在 `trading_date` 边界（如果是，说明 `contract` 在一个
trading_date 内恒定，后面按 trading_date 分组是安全的）。

In [17]:
roll = np.empty(len(df), dtype=bool)
roll[0] = False
roll[1:] = SAME_PREV[1:] & (con_codes[1:] != con_codes[:-1])

td_chg = np.empty(len(df), dtype=bool)
td_chg[0] = False
td_chg[1:] = TD_D[1:] != TD_D[:-1]

prev_close = np.empty(len(df))
prev_close[0] = np.nan
prev_close[1:] = c[:-1]
roll_gap = np.where(roll & (prev_close > 0), o / prev_close - 1.0, np.nan)

C9 = tbl(
    n_contract=pd.Series(con_codes).groupby(sym_codes).nunique().sort_index().to_numpy(),
    n_rollover=cnt(roll),
    roll_not_at_td_edge=cnt(roll & ~td_chg),
    roll_gap_abs_max=g(np.abs(roll_gap), "max"),
    roll_gap_abs_mean=g(np.abs(roll_gap), "mean"),
)
print("=" * 70, "\nC9 合约与换月\n", "=" * 70)
print(f"全表换月次数: {int(roll.sum()):,}")
print(f"不在 trading_date 边界上的换月: {int((roll & ~td_chg).sum()):,}  <- 0 才说明可以按 trading_date 分组")
print("\n--- 逐品种 ---")
print(C9.sort_values("roll_gap_abs_max", ascending=False).to_string())

print(f"\n--- {SYMS[rb]} 全部换月明细（最后 10 次）---")
m = roll & (sym_codes == rb)
print(pd.DataFrame({
    "at": df.index[m],
    "to": df["contract"].to_numpy()[m],
    "prev_close": prev_close[m],
    "new_open": o[m],
    "gap_pct": roll_gap[m] * 100,
}).tail(10).to_string(index=False))

C:\Users\Snowman\AppData\Local\Temp\ipykernel_40520\4150991023.py:12: RuntimeWarning: divide by zero encountered in divide
  roll_gap = np.where(roll & (prev_close > 0), o / prev_close - 1.0, np.nan)
C:\Users\Snowman\AppData\Local\Temp\ipykernel_40520\4150991023.py:12: RuntimeWarning: invalid value encountered in divide
  roll_gap = np.where(roll & (prev_close > 0), o / prev_close - 1.0, np.nan)


C9 合约与换月
全表换月次数: 848,065
不在 trading_date 边界上的换月: 842,548  <- 0 才说明可以按 trading_date 分组

--- 逐品种 ---
     n_contract  n_rollover  roll_not_at_td_edge  roll_gap_abs_max  roll_gap_abs_mean
sym                                                                                  
BB           33          98                    0          0.518048           0.064447
CJ           22          21                    0          0.462760           0.072994
FB           51          50                    0          0.352459           0.058306
JD           60          59                    0          0.351981           0.080539
AP           26          25                    0          0.332888           0.075380
LH           26          25                    0          0.321168           0.084607
ZC           56      242746               242430          0.240964           0.000008
RU           52          51                    0          0.233065           0.044020
WH           53      178222              

## C10 价格跳变异常

只在**真正相邻的两个分钟**之间比较（时间差恰好 60 秒），因此天然排除了
小节休息、隔夜、跨 session 的合法跳空；同时排除换月点。

剩下的大跳变就是真正需要怀疑的坏数据。

In [18]:
contig = (DT_NS == 60_000_000_000) & SAME_PREV
valid_ret = contig & ~roll & (prev_close > 0) & (c > 0)
ret = np.where(valid_ret, c / np.where(prev_close > 0, prev_close, np.nan) - 1.0, np.nan)
jump = valid_ret & (np.abs(ret) > TH_JUMP)

C10 = tbl(
    n_contiguous_pairs=cnt(valid_ret),
    n_jump=cnt(jump),
    ret_abs_max=g(np.abs(ret), "max"),
)
C10["jump_per_10k"] = C10["n_jump"] / C10["n_contiguous_pairs"] * 1e4

print("=" * 70, f"\nC10 相邻分钟跳变 > {TH_JUMP:.0%}\n", "=" * 70)
print(f"全表异常跳变: {int(jump.sum()):,}")
print("\n--- 逐品种（按每万对的异常率降序）---")
print(C10.sort_values("jump_per_10k", ascending=False).to_string())

if jump.any():
    print("\n--- 跳变最大的 20 行 ---")
    ji = np.flatnonzero(jump)
    ji = ji[np.argsort(-np.abs(ret[ji]))][:20]
    print(pd.DataFrame({
        "at": df.index[ji],
        "sym": SYMS[sym_codes[ji]],
        "contract": df["contract"].to_numpy()[ji],
        "prev_close": prev_close[ji],
        "close": c[ji],
        "ret_pct": ret[ji] * 100,
        "volume": v[ji],
    }).to_string(index=False))

C10 相邻分钟跳变 > 5%
全表异常跳变: 666

--- 逐品种（按每万对的异常率降序）---
     n_contiguous_pairs  n_jump  ret_abs_max  jump_per_10k
sym                                                       
BB               655841     135     0.172727      2.058426
RS               705349     133     0.203380      1.885591
EC                35234       6     0.101323      1.702901
FB               648261      73    14.934066      1.126090
WR               419240      32     0.119970      0.763286
LR               582839      43     0.162973      0.737768
JR               628149      40     0.099643      0.636792
PM               860433      45     0.151414      0.522992
LC                39694       2     0.077916      0.503854
SF               646477      26     0.088726      0.402180
ZC               942151      33     0.136364      0.350262
RI               860401      26     0.104596      0.302185
WH               832044      17     0.085714      0.204316
SM               646477       5     0.138819      0.077342
BU  

## C11 流动性

按 (品种, trading_date) 汇总成交量/成交额，再取品种自己的**中位数**（不用均值：
单日暴量会把均值拉飞）。持仓取每日最后一根 bar 的 `open_interest`。

这是「筛掉哪些品种」最直接的依据。

In [19]:
vol_by_ktd = np.bincount(KEY_TD, weights=v, minlength=NS * NTD)
turn_by_ktd = np.bincount(KEY_TD, weights=tn, minlength=NS * NTD)

daily = pd.DataFrame({
    "sym_code": nzk // NTD,
    "volume": vol_by_ktd[nzk],
    "turnover": turn_by_ktd[nzk],
})
# 每日持仓 = 该 (品种,trading_date) 最后一根 bar 的 open_interest
last_row_of_ktd = pd.Series(np.arange(len(df))).groupby(KEY_TD).max()
oi_daily = pd.Series(oi[last_row_of_ktd.to_numpy()], index=last_row_of_ktd.index)

liq = daily.groupby("sym_code").agg(
    med_daily_volume=("volume", "median"),
    med_daily_turnover=("turnover", "median"),
)
oi_med = oi_daily.groupby(oi_daily.index.to_numpy() // NTD).median()

C11 = tbl(
    med_daily_volume=liq["med_daily_volume"].reindex(range(NS)).to_numpy(),
    med_daily_turnover=liq["med_daily_turnover"].reindex(range(NS)).to_numpy(),
    med_daily_open_interest=oi_med.reindex(range(NS)).to_numpy(),
)
C11["illiquid"] = C11["med_daily_volume"] < TH_MIN_MEDIAN_VOLUME

print("=" * 70, "\nC11 流动性（日中位数）\n", "=" * 70)
print(C11.sort_values("med_daily_turnover", ascending=False).to_string())
print(f"\n流动性不足 (日成交中位数 < {TH_MIN_MEDIAN_VOLUME:,}): {int(C11['illiquid'].sum())}")

C11 流动性（日中位数）
     med_daily_volume  med_daily_turnover  med_daily_open_interest  illiquid
sym                                                                         
SC           120006.0        5.841095e+10                  29682.0     False
T             51169.5        5.145755e+10                  89321.5     False
RB          1441955.0        5.060010e+10                1273599.0     False
TS            22104.5        4.471525e+10                  40672.0     False
RU           284355.0        4.175661e+10                 147507.0     False
I            610512.0        4.074748e+10                 567861.0     False
AG           456152.5        3.271589e+10                 324301.5     False
SA           890695.0        3.174032e+10                 622104.5     False
CU           107736.0        3.140442e+10                 119621.0     False
NI           250419.0        2.745258e+10                 122210.0     False
AU            89131.0        2.734454e+10                 1309

## C12 行级坏数据掩码

**区分两个完全不同的问题，别混在一起：**

| 层级 | 性质 | 处置 |
|---|---|---|
| **品种级** | 结构性、修不了（vintage 截断、时间戳重复、K 不可用、流动性不足） | 剔掉整个 ticker |
| **行级** | 局部脏行（0 价、负量、负额、异常跳变、padding 假 bar） | 只丢那些行，品种留着 |

反例警示：负成交额命中了 22 个品种，其中 CF/MA/OI/RM/SR/TA/PB/SA/PF/PK/RU 都是主力
品种，坏行只有个位数到千位数（占比 < 0.1%）。**因为几行负成交额就剔掉螺纹钢级别的
品种是荒谬的** —— 这类必须走行级。

In [20]:
bad_px_row = (o <= 0) | (h <= 0) | (lo <= 0) | (c <= 0) | np.isnan(o) | np.isnan(c)
bad_ohlc_row = (h < lo) | (h < np.maximum(o, c)) | (lo > np.minimum(o, c))
bad_vol_row = (v < 0) | np.isnan(v)
bad_turn_row = tn < 0
contra_row = ((v == 0) & (tn > 0)) | ((v > 0) & (tn == 0))
fake_day_row = row_fake_cd                       # 落在「整日零成交」的假日历日里
pad_row = (o == h) & (h == lo) & (lo == c) & (v == 0)
jump_row = jump                                   # C10 的异常跳变

# 建议丢弃的行 = 价格不可信 + 假日历日 + 异常跳变
DROP_ROW = bad_px_row | bad_ohlc_row | bad_vol_row | fake_day_row | jump_row

C12 = tbl(
    n_bars=N_BARS,
    bad_px=cnt(bad_px_row),
    bad_ohlc=cnt(bad_ohlc_row),
    bad_vol=cnt(bad_vol_row),
    bad_turn=cnt(bad_turn_row),
    contradictory=cnt(contra_row),
    fake_day_rows=cnt(fake_day_row),
    pad_rows=cnt(pad_row),
    jump_rows=cnt(jump_row),
    drop_rows=cnt(DROP_ROW),
)
C12["drop_pct"] = C12["drop_rows"] / C12["n_bars"]

print("=" * 70, "\nC12 行级坏数据\n", "=" * 70)
print(f"建议丢弃行数: {int(DROP_ROW.sum()):,} / {len(df):,} ({DROP_ROW.mean():.2%})")
print(f"  其中 价格不可信 {int(bad_px_row.sum()):,} | OHLC违规 {int(bad_ohlc_row.sum()):,} | "
      f"负量 {int(bad_vol_row.sum()):,} | 假日历日 {int(fake_day_row.sum()):,} | "
      f"异常跳变 {int(jump_row.sum()):,}")
print(f"\n注：负成交额 {int(bad_turn_row.sum()):,} 行**未**计入 DROP_ROW —— "
      f"total_turnover 本轮不用，且不影响 OHLCV。要用时再单独处理。")
print("\n--- 逐品种（按丢弃占比降序，前 25）---")
print(C12.sort_values("drop_pct", ascending=False).head(25).to_string())

C12 行级坏数据
建议丢弃行数: 4,561,895 / 66,284,866 (6.88%)
  其中 价格不可信 96 | OHLC违规 0 | 负量 1 | 假日历日 4,561,132 | 异常跳变 666

注：负成交额 4,160 行**未**计入 DROP_ROW —— total_turnover 本轮不用，且不影响 OHLCV。要用时再单独处理。

--- 逐品种（按丢弃占比降序，前 25）---
      n_bars  bad_px  bad_ohlc  bad_vol  bad_turn  contradictory  fake_day_rows  pad_rows  jump_rows  drop_rows  drop_pct
sym                                                                                                                      
LR    760038       0         0        0       287              6         618336    730142         43     618379  0.813616
JR    815634       0         0        0       469              1         639128    803521         40     639168  0.783646
PM    909198       0         0        0       212              1         649750    874029         45     649795  0.714690
BB    664666       1         0        0         0              0         421264    581374        135     421400  0.634003
RI    908972       0         0        0       585        

## ★ 汇总打分表 + 建议剔除名单

把 C1-C12 的结论合成一张表。**品种级 kill 判据**（任一命中即建议剔除）：

1. `is_stale` —— vintage 截断，历史残缺
2. `too_short` —— 交易日 < `TH_MIN_TRADING_DAYS`
3. `dup_exact > 0` —— 同品种内重复时间戳，索引不变式被破坏
4. `k_bad_pct > TH_K_BAD_PCT` —— 无法复权
5. `fake_day_pct > TH_FAKE_DAY_PCT` —— 大半"历史"是补出来的
6. `illiquid` —— 日成交中位数不足

改上面的阈值再重跑这一格即可。

In [21]:
S = pd.concat([
    C2[["n_bars", "first_ts", "last_ts", "n_trading_days", "stale_days", "is_stale", "too_short"]],
    C1[["dup_exact", "time_backwards"]],
    C3[["nonpos_px", "high_lt_low", "high_not_max", "low_not_min", "ohlc_bad_total"]],
    C4[["vol_neg", "turn_neg", "contradictory", "vol_zero_pct"]],
    C5[["k_bad", "k_bad_pct", "k_min", "k_max"]],
    C6[["pad_pct", "fake_cal_days", "fake_day_pct"]],
    C7[["mode_cal_days", "td_polluted_by_fake_day", "td_polluted_pct"]],
    C8[["n_sessions", "has_night", "crosses_midnight", "fingerprint"]],
    shift_tbl[["n_distinct"]].rename(columns={"n_distinct": "bars_per_day_variants"}),
    C9[["n_contract", "n_rollover", "roll_not_at_td_edge", "roll_gap_abs_max"]],
    C10[["n_jump", "jump_per_10k"]],
    C11[["med_daily_volume", "med_daily_turnover", "med_daily_open_interest", "illiquid"]],
    C12[["drop_rows", "drop_pct"]],
], axis=1)

# ---- 品种级 kill 判据（结构性、修不了）----
KILL = {
    "停更": S["is_stale"],
    "历史过短": S["too_short"],
    "重复时间戳": S["dup_exact"] > 0,
    "K不可用": S["k_bad_pct"] > TH_K_BAD_PCT,
    "多半是假日": S["fake_day_pct"] > TH_FAKE_DAY_PCT,
    "流动性不足": S["illiquid"],
}
for name, m in KILL.items():
    S[f"kill_{name}"] = m.fillna(False).astype(bool)

detail = {
    "停更": lambda r: f"停更{int(r['stale_days'])}天",
    "历史过短": lambda r: f"仅{int(r['n_trading_days'])}交易日",
    "重复时间戳": lambda r: f"重复时间戳{int(r['dup_exact']):,}根",
    "K不可用": lambda r: f"K坏值{r['k_bad_pct']:.1%}",
    "多半是假日": lambda r: f"假日历日{r['fake_day_pct']:.0%}",
    "流动性不足": lambda r: f"日成交中位数{r['med_daily_volume']:.0f}",
}
S["drop_reason"] = [
    "; ".join(detail[n](r) for n, m in KILL.items() if bool(m.fillna(False).loc[s]))
    for s, r in S.iterrows()
]
S["KEEP"] = S["drop_reason"] == ""

print("=" * 70, "\n★ 汇总打分表\n", "=" * 70)
print(f"品种总数: {len(S)}   建议保留: {int(S['KEEP'].sum())}   建议剔除: {int((~S['KEEP']).sum())}")
print("\n--- 各 kill 判据分别命中几个品种（可重叠）---")
for name, m in KILL.items():
    hit = sorted(S.index[m.fillna(False)].tolist())
    print(f"  {name:<8} {len(hit):>3}  {' '.join(hit)}")

print("\n--- 建议剔除 ---")
print(S.loc[~S["KEEP"], ["n_bars", "n_trading_days", "last_ts", "med_daily_volume", "drop_reason"]]
      .sort_values("n_bars", ascending=False).to_string())
print("\n--- 建议保留（按日成交额中位数降序）---")
keep_cols = ["n_bars", "n_trading_days", "last_ts", "n_sessions", "has_night", "crosses_midnight",
             "med_daily_volume", "med_daily_turnover", "fake_cal_days", "n_rollover",
             "drop_rows", "drop_pct"]
print(S.loc[S["KEEP"], keep_cols].sort_values("med_daily_turnover", ascending=False).to_string()
      if S["KEEP"].any() else "  (无)")

print("\n--- 保留品种里仍需**行级**清洗的（按坏行占比降序）---")
rowlvl = S.loc[S["KEEP"], ["drop_rows", "drop_pct", "nonpos_px", "vol_neg", "turn_neg",
                           "contradictory", "fake_cal_days", "n_jump"]]
print(rowlvl[rowlvl["drop_rows"] > 0].sort_values("drop_pct", ascending=False).to_string())

print("\n--- 可直接粘贴使用 ---")
print(f"KEEP_SYMBOLS = {sorted(S.index[S['KEEP']].tolist())}")
print(f"DROP_SYMBOLS = {sorted(S.index[~S['KEEP']].tolist())}")

★ 汇总打分表
品种总数: 79   建议保留: 57   建议剔除: 22

--- 各 kill 判据分别命中几个品种（可重叠）---
  停更        18  AO BB BR EC FB IM JR LC LR PM PX RI RS SH SI TL WH ZC
  历史过短       7  AO BR EC LC PX SH TL
  重复时间戳      6  JR LR PM RI WH ZC
  K不可用       6  JR PM RI RS WH ZC
  多半是假日      9  BB FB JR LR PM RI RS WH ZC
  流动性不足     12  BB BC CY FB JR LR PM RI RR RS WH WR

--- 建议剔除 ---
      n_bars  n_trading_days             last_ts  med_daily_volume                                          drop_reason
sym                                                                                                                    
ZC   1195605            2987 2026-01-16 15:00:00           30198.0             停更194天; 重复时间戳206,216根; K坏值40.9%; 假日历日28%
WH   1021294            3894 2026-01-16 15:00:00             295.5  停更194天; 重复时间戳141,250根; K坏值33.7%; 假日历日23%; 日成交中位数296
PM    909198            3894 2026-01-16 15:00:00               0.0     停更194天; 重复时间戳29,154根; K坏值24.6%; 假日历日71%; 日成交中位数0
RI    908972            3894 2026-01-16 15:00:

In [22]:
# ============================ 落盘报告（可选）============================
S.to_csv(rf"{OUTDIR}\audit_scorecard.csv")
C8.join(C8_all[["fingerprint"]].rename(columns={"fingerprint": "fingerprint_all_history"})) \
    .to_csv(rf"{OUTDIR}\audit_C8_session_fingerprint.csv")
C8b.to_csv(rf"{OUTDIR}\audit_C8b_bars_per_day_by_year.csv")
per_td.to_csv(rf"{OUTDIR}\audit_bars_per_trading_date.csv", index=False)
print("已写出:")
print(f"  {OUTDIR}\\audit_scorecard.csv                  <- 主报告，逐品种全部指标 + KEEP/drop_reason")
print(f"  {OUTDIR}\\audit_C8_session_fingerprint.csv      <- 每品种 session 指纹(当前 + 全历史)")
print(f"  {OUTDIR}\\audit_C8b_bars_per_day_by_year.csv    <- 逐年 bar 数换挡")
print(f"  {OUTDIR}\\audit_bars_per_trading_date.csv       <- 每(品种,交易日) bar 数明细")

已写出:
  D:\2026_Summer\TradingApp\data\v3.0\audit_scorecard.csv                  <- 主报告，逐品种全部指标 + KEEP/drop_reason
  D:\2026_Summer\TradingApp\data\v3.0\audit_C8_session_fingerprint.csv      <- 每品种 session 指纹(当前 + 全历史)
  D:\2026_Summer\TradingApp\data\v3.0\audit_C8b_bars_per_day_by_year.csv    <- 逐年 bar 数换挡
  D:\2026_Summer\TradingApp\data\v3.0\audit_bars_per_trading_date.csv       <- 每(品种,交易日) bar 数明细
